# Download via Library

This notebook demonstrates how to use `media-downloader` directly as a Python library without running any server.

Steps:

1. Choose a URL (YouTube, TikTok, Instagram, or a recipe page)
2. Build `MediaStorage` to control where files land on disk
3. Build a `DownloadRouter` with the providers you want
4. Call `router.download(url)` and inspect the `DownloadedMedia` result


## Imports


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
import asyncio
from pathlib import Path

from rich import get_console
from rich import print as rprint
from rich.console import Console

from media_downloader.core.models import DownloadedMedia
from media_downloader.core.providers.instagram import InstaDownloader
from media_downloader.core.providers.web_recipe import WebRecipeDownloader
from media_downloader.core.providers.yt_dlp import YtDlpDownloader
from media_downloader.core.router import DownloadRouter
from media_downloader.storage.media_storage import MediaStorage

# make rich work in jupyter notebooks
console: Console = get_console()
console.is_jupyter = False


## Configuration

Set the output directory and the URL you want to download.


In [ ]:
OUTPUT_DIR = Path("tmp/media-downloader-demo")

# Change this to whatever you want to download

# URL = "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
URL = "https://www.instagram.com/reel/DVvd_q1CsWE/"


## Build storage and router

`MediaStorage` controls the directory layout: `{output_dir}/{source}/{source_id}/`.

`DownloadRouter` accepts any combination of providers. The URL detector picks the right one automatically.


In [ ]:
storage = MediaStorage(base_dir=OUTPUT_DIR)

router = DownloadRouter(
    downloaders=[
        YtDlpDownloader(storage=storage),
        WebRecipeDownloader(storage=storage),
        InstaDownloader(storage=storage),
    ]
)

print(f"Storage root: {storage.base_dir}")
print(f"Registered sources: {list(router._source_map.keys())}")  # noqa: SLF001


## Download a URL

Call `router.download(url)` to run the full pipeline: detect source, download, run post-processors.


In [ ]:
media = router.download(URL)
rprint(media)


## Inspect the result

`DownloadedMedia` carries everything the pipeline produced.


In [ ]:
print("Source:       ", media.source.value)
print("Source ID:    ", media.source_id)
print("Original URL: ", media.original_url)
print("Caption:      ", (media.caption or "")[:120])
print()
print("Video file:   ", media.video_file)
print("Thumbnail:    ", media.thumbnail_file)
print("Audio file:   ", media.audio_file)
print()
print("All files:")
for f in media.all_files:
    print(f"  {f.relative_to(Path().cwd())}")
print()
print("Transcription:", media.transcript)
print("Language:     ", media.language)


## Provider-specific metadata

Each provider populates `media.metadata` with structured data.


In [ ]:
if media.metadata is not None:
    rprint(media.metadata)
else:
    print("No metadata (provider does not supply structured metadata for this source)")


## Web recipe example

Recipe pages produce a `WebRecipeMetadata` object with structured fields.


In [ ]:
RECIPE_URL = "https://www.allrecipes.com/recipe/10813/best-chocolate-chip-cookies/"

recipe_media = router.download(RECIPE_URL)

print("Source:   ", recipe_media.source.value)
print("Caption preview:\n", (recipe_media.caption or "")[:500])


In [ ]:
if recipe_media.metadata is not None:
    rprint(recipe_media.metadata)


## Async download

Use `router.adownload(url)` in async contexts (e.g. inside a FastAPI handler or `asyncio.run()`).


In [ ]:
async def download_async(url: str) -> DownloadedMedia:
    """Show an example of how to call the async download method."""
    return await router.adownload(url)


# In a regular Jupyter cell, use asyncio.run() or await directly
async_media = await download_async(URL)
print("Async result:", async_media.source.value, async_media.source_id)
